In [10]:
"""
SMBHB Doppler-boost detectability: max redshift for a FIXED orbital period,
requiring the periodic amplitude to clear a photometric-noise threshold.

============================= THE QUESTION =============================
Given a binary of total mass M_tot, radiating at Eddington ratio lambda_Edd,
with a FIXED observed orbital period P_obs = 2 yr, how far away (what z_max)
could ASAS-SN / ATLAS / ZTF still detect the Doppler-boost periodic
modulation, requiring the amplitude to exceed n_sigma times the survey's
photometric uncertainty AT THE APPARENT MAGNITUDE THE SOURCE ACTUALLY HAS?

This is a SINGLE-EPOCH-EQUIVALENT significance test (no credit for
averaging over many epochs) -- deliberately conservative, meant as a
first-pass cut before a full DRW/red-noise false-alarm simulation.

============================= ASSUMPTIONS ===============================
1. Binary/AGN physical parameters (all fiducial -- edit freely):
   - lambda_Edd = 0.1        Eddington ratio
   - BC = 10                 bolometric correction to 5100 A (Richards+2006)
   - q = 0.5                 mass ratio M2/M1
   - alpha = 0.5             optical spectral index F_nu ~ nu^-alpha
                             (Vanden Berk et al. 2001 measure ~0.44+-0.10;
                              0.5 is a round approximation to that)
   - inc_deg = 45             inclination (degrees)
   - f_supp = 1.0            accretion suppression factor (1 = no suppression;
                             set <1 to model binary-specific accretion dip)
   - n_sigma = 5             required amplitude significance

2. Doppler-boost amplitude formula (D'Orazio et al. 2015-style):
      beta = v2/c = 0.0461 * (Mtot/1e8 Msun)^(1/3) * (P_rest/1yr)^(-1/3) / (1+q)
      Delta_F/F = (3-alpha) * beta * sin(i)
      Delta_m = 1.086 * Delta_F/F      [small-amplitude log-linearization]

3. Baseline (time-averaged) absolute magnitude:
      L_bol = f_supp * lambda_Edd * 1.26e38 erg/s * (Mtot/Msun)     [Eddington]
      M_AB0 = -26.73 - 2.5*log10[ f_supp*(lambda_Edd/BC)*(Mtot/1e8) ]
   (derivation: L_bol -> L_band=L_bol/BC -> L_nu=L_band/nu -> AB mag
    via M_AB = -2.5log10(L_nu) + 51.6, itself derived from the Oke & Gunn
    1983 apparent-AB-mag zero point -48.60 placed at a 10 pc reference
    distance -- see the walkthrough of this exact derivation.)

4. Cosmology: flat LambdaCDM, H0=70, Om0=0.3 (Hogg 1999 for the general
   distance formalism; these are just round fiducial parameter values,
   not a specific measurement).

5. K-correction for a power-law source (Hogg et al. 2002):
      K(z) = -2.5*(1-alpha)*log10(1+z)

6. Photometric uncertainty vs magnitude, sigma(m) = sigma0 * 10^(k*(m-m1)),
   fit through two SOURCED anchor points per survey:
      ASAS-SN: 0.02 mag @ m=14, 0.16 mag @ m=18   (Jayasinghe et al. 2019)
      ZTF:     0.01 mag @ m=14, 0.15 mag @ m=20.6 (Masci et al. 2019 and
                                                    several papers citing it)
      ATLAS:   0.015 mag @ m=14, 0.15 mag @ m=19.5 (INTERPOLATED, not
                                                     directly sourced --
                                                     weakest link here)

7. Survey single-epoch 5-sigma limiting magnitudes (per-visit depth):
      ASAS-SN g ~ 17.5   (Jayasinghe et al. 2018)
      ATLAS o/c ~ 19.5   (Tonry et al. 2018a)
      ZTF r ~ 20.5       (Bellm et al. 2019)

============================= WHAT'S NOT INCLUDED =======================
- No epoch-averaging / Lomb-Scargle sqrt(N/2) boost (deliberately, for
  conservatism -- see the earlier white-noise-periodogram version of this
  script if you want that included instead).
- No DRW/red-noise false-alarm correction -- a candidate clearing this
  cut still needs the simulation-based test to be a genuine detection
  claim, since real AGN variability is NOT simply white photometric noise.
- No accretion-partition dilution (f2 = fraction of light in the
  modulated component) -- multiply Delta_m by f2 if you want to include it.
"""

import numpy as np
from astropy.cosmology import FlatLambdaCDM
from scipy.optimize import brentq

cosmo = FlatLambdaCDM(H0=70, Om0=0.3)

# ================================================================
# 1) PHOTOMETRIC UNCERTAINTY CURVES  sigma(m) = sigma0 * 10^(k*(m-m1))
#    fit through two sourced (m, sigma) anchor points
# ================================================================
def make_sigma_func(m1, sigma1, m2, sigma2):
    k = np.log10(sigma2 / sigma1) / (m2 - m1)
    return lambda m: sigma1 * 10 ** (k * (m - m1))

SIGMA_FUNCS = {
    "ASAS-SN": make_sigma_func(14, 0.02, 17.5, 0.16),      # Jayasinghe et al. 2019
    "ZTF":     make_sigma_func(14, 0.01, 20.8, 0.15),    # Masci et al. 2019 et seq.
    "ATLAS":   make_sigma_func(14, 0.015, 19.5, 0.15),   # interpolated, weakest link
}
M_LIM = {"ASAS-SN": 17.5, "ATLAS": 19.5, "ZTF": 20.5}    # single-epoch 5-sigma limits

# ================================================================
# 2) FIDUCIAL PHYSICAL PARAMETERS -- edit these to explore other cases
# ================================================================
P_OBS_YR = 2.0        # FIXED observed orbital period
Q        = 0.3        # mass ratio M2/M1
ALPHA    = 0.5        # optical spectral index (Vanden Berk et al. 2001: ~0.44)
INC_DEG  = 45.0        # inclination
LAM_EDD  = 0.1        # Eddington ratio
BC       = 10.0       # bolometric correction (Richards et al. 2006)
F_SUPP   = 1.0        # binary accretion suppression factor (1 = none)
N_SIGMA  = 5.0        # required amplitude significance
MTOT_LIST = [1e7, 1e8, 3.16e8, 1e9]   # Msun, swept
M_chirp = 3e9
Q = 0.5
q = Q
Mtot = M_chirp * (1 + q) ** (1 / 5) / q ** (3 / 5)
MTOT_LIST = [Mtot]
I_RAD = np.radians(INC_DEG)

# ================================================================
# 3) CORE PHYSICS FUNCTIONS
# ================================================================
def M_AB0(Mtot, lam_Edd=LAM_EDD, BC=BC, f_supp=F_SUPP):
    """Baseline (time-averaged) absolute AB magnitude at 5100 A."""
    return -26.73 - 2.5 * np.log10(f_supp * (lam_Edd / BC) * (Mtot / 1e8))

def beta_doppler(Mtot, P_rest_yr, q=Q):
    """Orbital velocity of the secondary about the barycenter, in units of c."""
    return 0.0461 * (Mtot / 1e8) ** (1 / 3) * P_rest_yr ** (-1 / 3) / (1 + q)

def delta_m(Mtot, P_rest_yr, q=Q, alpha=ALPHA, i=I_RAD):
    """Doppler-boost peak-to-mean magnitude amplitude."""
    beta = beta_doppler(Mtot, P_rest_yr, q)
    dF_F = (3 - alpha) * beta * np.sin(i)
    return 1.086 * dF_F

def m_apparent(Mabs, z, alpha=ALPHA):
    """Apparent magnitude at redshift z: distance modulus + K-correction."""
    if z <= 0:
        return -np.inf
    DL_Mpc = cosmo.luminosity_distance(z).value
    DM = 5 * np.log10(DL_Mpc * 1e6 / 10)
    K = -2.5 * (1 - alpha) * np.log10(1 + z)
    return Mabs + DM + K

# ================================================================
# 4) SOLVE FOR z_max: largest z where Delta_m(z) >= n_sigma * sigma(m_app(z)),
#    subject to the source still being brighter than the survey's flux limit
# ================================================================
def z_max_amplitude(Mtot, sigma_func, mlim, P_obs_yr=P_OBS_YR,
                     q=Q, alpha=ALPHA, i=I_RAD, n_sigma=N_SIGMA,
                     zmax_search=5.0):
    Mabs = M_AB0(Mtot)

    def f(z):
        z = max(z, 1e-4)
        P_rest = P_obs_yr / (1 + z)          # time dilation: rest period shrinks with z
        dm = delta_m(Mtot, P_rest, q, alpha, i)
        m_app = m_apparent(Mabs, z, alpha)
        if m_app > mlim:                      # fainter than survey's detection floor
            return -1e6
        sig = sigma_func(m_app)
        return dm - n_sigma * sig

    if f(1e-4) < 0:
        return 0.0, np.nan, np.nan            # fails even at z~0
    if f(zmax_search) > 0:
        return zmax_search, np.nan, np.nan    # still passes at search boundary
    z_amp = brentq(f, 1e-4, zmax_search)
    m_app = m_apparent(Mabs, z_amp, alpha)
    dm = delta_m(Mtot, P_obs_yr / (1 + z_amp), q, alpha, i)
    return z_amp, m_app, dm

# ================================================================
# 5) RUN AND PRINT
# ================================================================
if __name__ == "__main__":
    print(f"Fixed P_obs = {P_OBS_YR} yr, q={Q}, i={INC_DEG} deg, "
          f"n_sigma={N_SIGMA}, lambda_Edd={LAM_EDD}, BC={BC}\n")
    for survey in ["ASAS-SN", "ATLAS", "ZTF"]:
        print(f"=== {survey} (m_lim={M_LIM[survey]}) ===")
        print(f"{'Mtot [Msun]':>12} {'z_max':>8} {'D_L [Mpc]':>10} "
              f"{'m_app':>8} {'Delta_m':>9} {'sigma(m)':>9}")
        for Mtot in MTOT_LIST:
            z_amp, m_app, dm = z_max_amplitude(Mtot, SIGMA_FUNCS[survey], M_LIM[survey])
            sig = SIGMA_FUNCS[survey](m_app) if not np.isnan(m_app) else np.nan
            DL = cosmo.luminosity_distance(z_amp).value if z_amp > 0 else 0.0
            print(f"{Mtot:12.2e} {z_amp:8.4f} {DL:10.2f} {m_app:8.2f} {dm:9.4f} {sig:9.4f}")
        print()

Fixed P_obs = 2.0 yr, q=0.5, i=45.0 deg, n_sigma=5.0, lambda_Edd=0.1, BC=10.0

=== ASAS-SN (m_lim=17.5) ===
 Mtot [Msun]    z_max  D_L [Mpc]    m_app   Delta_m  sigma(m)
    4.93e+09   0.3289    1726.96    15.07    0.1888    0.0378

=== ATLAS (m_lim=19.5) ===
 Mtot [Msun]    z_max  D_L [Mpc]    m_app   Delta_m  sigma(m)
    4.93e+09   0.5546    3209.19    16.33    0.1989    0.0398

=== ZTF (m_lim=20.5) ===
 Mtot [Msun]    z_max  D_L [Mpc]    m_app   Delta_m  sigma(m)
    4.93e+09   0.9593    6277.23    17.66    0.2149    0.0430

